In [3]:
import os
from typing import Annotated, Literal, List, Dict
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import  load_dotenv
import operator

In [4]:
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGSMITH_PROJECT")

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
class QuantumContext(BaseModel):
    """Stores the quantum state and circuit architecture."""
    qubit_count: int = Field(default=0)
    bloch_sphere_vectors: List[List[float]] = Field(default_factory=list)
    circuit_gates: List[str] = Field(default_factory=list) # e.g., ["H(0)", "CX(0,1)"]

class ClassicalContext(BaseModel):
    """Stores the classical matrices and optimization metrics."""
    input_dataset: List[List[float]] = Field(default_factory=list)
    current_loss: float = Field(default=float('inf'))
    rotation_gradients: List[float] = Field(default_factory=list)

class HybridResearchState(BaseModel):
    """The master global state orchestrating the 4th-semester research workflow."""
    messages: Annotated[list[BaseMessage], add_messages]
    quantum_data: QuantumContext = Field(default_factory=QuantumContext)
    classical_data: ClassicalContext = Field(default_factory=ClassicalContext)
    next_action: str = ""

In [ ]:
class RoutingDecision(BaseModel):
    next_action: Literal["StateEncoder", "EntanglementArchitect", "ParameterOptimizer", "FINISH"]

def supervisor_node(state: HybridResearchState) -> dict:
    prompt = (
        "You manage a hybrid quantum-classical machine learning workflow.\n"
        "- If classical data needs to be mapped to Bloch sphere angles, route to StateEncoder.\n"
        "- If quantum gates (Hadamard, CNOT) need to be designed, route to EntanglementArchitect.\n"
        "- If the circuit has been measured and gradients need calculating, route to ParameterOptimizer.\n"
        "- If the loss is minimized, route to FINISH."
    )
    messages = [{"role": "system", "content": prompt}] + state.messages
    decision = llm.with_structured_output(RoutingDecision).invoke(messages)
    
    print(f"[SUPERVISOR] Routing to: {decision.next_action}")
    return {"next_action": decision.next_action}

In [ ]:
# 3. SPECIALIZED WORKERS
# ==========================================
def state_encoder_node(state: HybridResearchState) -> dict:
    print("[ENCODER] Mapping classical dataset to rotational angles...")
    # Read classical, write to quantum
    angles = [0.5, 1.57] # Mock angles derived from classical input
    new_q_data = state.quantum_data.model_copy()
    new_q_data.bloch_sphere_vectors = [angles]
    return {
        "quantum_data": new_q_data,
        "messages": [AIMessage(content="Encoded classical data to Bloch vectors.", name="StateEncoder")]
    }

def entanglement_architect_node(state: HybridResearchState) -> dict:
    print("[ARCHITECT] Designing quantum circuit gates...")
    new_q_data = state.quantum_data.model_copy()
    new_q_data.circuit_gates = ["RX(0)", "RY(1)", "CNOT(0,1)"]
    return {
        "quantum_data": new_q_data,
        "messages": [AIMessage(content="Circuit generated with RX, RY, and CNOT gates.", name="EntanglementArchitect")]
    }

def parameter_optimizer_node(state: HybridResearchState) -> dict:
    print("[OPTIMIZER] Calculating classical gradients from circuit measurements...")
    new_c_data = state.classical_data.model_copy()
    new_c_data.current_loss = 0.05 # Mock optimized loss
    return {
        "classical_data": new_c_data,
        "messages": [AIMessage(content="Calculated gradients. Loss is minimized.", name="ParameterOptimizer")]
    }